In [1]:
# 1. IMPORT LIBRARIES
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text preprocessing
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Sklearn utilities
from sklearn.model_selection import train_test_split

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
project_path = '/content/drive/MyDrive/Fake News Project'
os.makedirs(project_path, exist_ok=True)

In [14]:

# 2. LOAD DATASET

# Load datasets
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Fake News Project/data/training_data.csv", delimiter = "\t", header = None)
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Fake News Project/data/testing_data.csv" , delimiter = "\t", header = None)


In [15]:
# DISPLAY BASIC INFORMATION

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape:", test_df.shape)


# Display first 5 rows
train_df.head()

Training Dataset Shape: (34152, 2)
Testing Dataset Shape: (9984, 2)


,0,1
0,0,donald trump sends out embarrassing new year‚s...
1,0,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...
3,0,trump is so obsessed he even has obama‚s name ...
4,0,pope francis just called out donald trump duri...


In [21]:
# Renamed the columns as label and text
train_df.columns = ['label', 'text']
test_df.columns = ['label', 'text']
train_df
test_df

,label,text
0,2,copycat muslim terrorist arrested with assault...
1,2,wow! chicago protester caught on camera admits...
2,2,germany's fdp look to fill schaeuble's big shoes
3,2,mi school sends welcome back packet warning ki...
4,2,u.n. seeks 'massive' aid boost amid rohingya '...
...,...,...
9979,2,boom! fox news leftist chris wallace attempts ...
9980,2,here it is: list of democrat hypocrites who vo...
9981,2,new fires ravage rohingya villages in northwes...
9982,2,meals on wheels shuts the lyin‚ lefties up wit...


In [22]:
# 3. EXPLORATORY DATA ANALYSIS (EDA)


# CHECK DATASET INFO


print("\nTraining Dataset Info:\n")
train_df.info()


Training Dataset Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34152 entries, 0 to 34151
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   34152 non-null  int64 
 1   text    34152 non-null  object
dtypes: int64(1), object(1)
memory usage: 533.8+ KB


In [23]:
# CHECK MISSING VALUES


print("\nMissing Values in Training Dataset:\n")
print(train_df.isnull().sum())


print("\nMissing Values in Testing Dataset:\n")
print(test_df.isnull().sum())


Missing Values in Training Dataset:

label    0
text     0
dtype: int64

Missing Values in Testing Dataset:

label    0
text     0
dtype: int64


In [24]:
# CHECK DUPLICATES


print("\nDuplicate Rows in Training Dataset:", train_df.duplicated().sum())
print("Duplicate Rows in Testing Dataset:", test_df.duplicated().sum())


Duplicate Rows in Training Dataset: 1946
Duplicate Rows in Testing Dataset: 775


In [26]:
train_df['text'].duplicated().sum()


np.int64(1946)

In [27]:
test_df['text'].duplicated().sum()

np.int64(776)

In [30]:
train_df[train_df['text'].duplicated()]

,label,text
1534,0,mcconnell says he‚ll obstruct any effort to hi...
3054,0,no
4047,0,trump freaks out
4963,0,as trump collapses
5141,0,no
...,...,...
34047,1,u.s. has lost trust in south sudan\ttrump envo...
34049,1,u.s. house passes sanctions on iran-backed hez...
34054,1,trump declines to say if he will visit korean ...
34126,1,thailand kicks off sumptuous funeral of king b...


In [31]:
train_df[train_df['text'].duplicated(keep = False)]

,label,text
678,0,thanks to trump
986,0,no
1487,0,mcconnell says he‚ll obstruct any effort to hi...
1534,0,mcconnell says he‚ll obstruct any effort to hi...
1619,0,trump freaks out
...,...,...
34049,1,u.s. house passes sanctions on iran-backed hez...
34053,1,thailand kicks off sumptuous funeral of king b...
34054,1,trump declines to say if he will visit korean ...
34126,1,thailand kicks off sumptuous funeral of king b...


In [32]:
# REMOVE DUPLICATES (OPTIONAL)


train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)

print("\nShape After Removing Duplicates:")
print("Training:", train_df.shape)
print("Testing:", test_df.shape)



Shape After Removing Duplicates:
Training: (32206, 2)
Testing: (9209, 2)


DATA CLEANING

In [34]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [35]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

In [36]:
def clean_text(text):

    # lowercase
    text = text.lower()

    # remove punctuation
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # tokenize
    words = text.split()

    # remove stopwords + lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [39]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

APPLY PREPROCESSING

In [40]:
train_df['text'] = train_df['text'].apply(clean_text)

In [41]:
test_df['text'] = test_df['text'].apply(clean_text)

In [42]:
train_df.head()
test_df.head()

,label,text
0,2,copycat muslim terrorist arrested assault weapon
1,2,wow chicago protester caught camera admits vio...
2,2,germany fdp look fill schaeuble big shoe
3,2,mi school sends welcome back packet warning ki...
4,2,u n seek massive aid boost amid rohingya emerg...


TRAIN-TEST SPLIT

In [45]:
from sklearn.model_selection import train_test_split

train_df_train, train_df_val = train_test_split(train_df, test_size = 0.2, random_state = 42, stratify = train_df['label'])

BAG OF WORDS

In [46]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

train_df_train_bow = vectorizer.fit_transform(train_df_train['text'])
train_df_val_bow = vectorizer.transform(train_df_val['text'])

# Print shape of vectorized dataset
print("Training BOW Shape:", train_df_train_bow.shape)
print("Validation BOW Shape:", train_df_val_bow.shape)

### then we continue on our classifier of choice

Training BOW Shape: (25764, 15165)
Validation BOW Shape: (6442, 15165)


TF-IDF

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

# Vectorize all dataset
train_df_train_tfidf = vectorizer.fit_transform(train_df_train['text'])
train_df_val_tfidf = vectorizer.transform(train_df_val['text'])

# Print shape of vectorized dataset
print("Training TF-IDF Shape:", train_df_train_tfidf.shape)
print("Validation TF-IDF Shape:", train_df_val_tfidf.shape)

### then we continue on our classifier of choice

Training TF-IDF Shape: (25764, 15165)
Validation TF-IDF Shape: (6442, 15165)
